### Chunking

In [36]:
import nltk
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Akash\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [37]:
documents = [

"""
Python is widely used in data science and machine learning.
Pandas helps analysts work with structured tabular datasets.
NumPy provides efficient numerical computations using arrays.
It is heavily used in scientific computing applications.
TensorFlow and PyTorch are widely used deep learning frameworks.
They help researchers build neural network models efficiently.

Climate change affects ecosystems and global temperatures.
Carbon emissions are one of the major causes of global warming.
Renewable energy sources such as solar and wind power reduce dependence on fossil fuels.
Governments are investing in sustainable transportation and clean energy systems.
""",

"""
The Taj Mahal is located in Agra, India.
It was built by Emperor Shah Jahan in memory of Mumtaz Mahal.
The monument attracts millions of tourists every year.
It is made primarily of white marble.

Paris is the capital city of France.
The Eiffel Tower attracts visitors from around the world.
The Louvre Museum contains famous artworks including the Mona Lisa.
French cuisine includes croissants, baguettes, and cheese dishes.
""",

"""
Football is one of the most popular sports worldwide.
The FIFA World Cup is organized every four years.
Lionel Messi and Cristiano Ronaldo are legendary football players.
Teams often rely on counterattacks and possession strategies.

Basketball requires teamwork, passing, and shooting accuracy.
The NBA is considered the most famous basketball league globally.
Michael Jordan inspired generations of basketball players.
Three-point shooting has become crucial in modern basketball tactics.
""",

"""
Space exploration has improved significantly in recent decades.
NASA launched several missions to study Mars and nearby planets.
Satellites help scientists monitor weather systems and communications.
Astronauts undergo years of physical and technical training.

Cybersecurity protects computer systems from malware and hacking attacks.
Encryption secures sensitive information shared online.
Hackers often exploit weak passwords and vulnerable software systems.
Organizations use authentication and firewalls to improve digital security.
""",

"""
Artificial intelligence is transforming healthcare and finance industries.
Machine learning systems can analyze large datasets efficiently.
Hospitals use AI tools to assist doctors in disease diagnosis.
Financial institutions use AI for fraud detection and risk analysis.

Ocean pollution threatens marine ecosystems worldwide.
Plastic waste harms fish, turtles, and coral reefs.
Environmental organizations promote recycling and waste reduction programs.
Governments are introducing laws to reduce single-use plastics.
"""
]

queries = [
    ("Which technology helps build neural network models?", "TensorFlow"),
    ("What reduces dependence on fossil fuels?", "Renewable energy"),
    ("Who built the Taj Mahal?", "Shah Jahan"),
    ("Which museum contains the Mona Lisa?", "Louvre"),
    ("Which basketball league is globally famous?", "NBA"),
    ("What protects systems from malware attacks?", "Cybersecurity"),
    ("What secures sensitive online information?", "Encryption"),
    ("Which library supports scientific numerical computation?", "NumPy"),
    ("What helps doctors diagnose diseases?", "AI"),
    ("What harms marine ecosystems?", "Plastic waste")
]

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

Fixed-Size chunking

In [38]:
def fixed_size_chunking(text,
                        chunk_size=25,
                        overlap=5):
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = words[i:i + chunk_size]
        chunks.append(" ".join(chunk))
        
    return chunks

Sentence-Aware Chunking

In [39]:
from nltk.tokenize import sent_tokenize

def sentence_aware_chunking(text,
                            max_words=25,
                            overlap_sentences=1):
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_len = 0
    for sentence in sentences:
        sentence_len = len(sentence.split())
        if current_len + sentence_len > max_words:
            chunks.append(" ".join(current_chunk))
            overlap = current_chunk[-overlap_sentences:]
            current_chunk = overlap.copy()
            current_len = sum(
                len(s.split()) for s in current_chunk
            )
        current_chunk.append(sentence)
        current_len += sentence_len
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks


Semantic chunking

In [40]:
from sklearn.metrics.pairwise import cosine_similarity

def semantic_chunking(text,
                      similarity_threshold=0.40,
                      min_sentences=2,
                      max_words=40):

    sentences = sent_tokenize(text)

    embeddings = embed_model.encode(sentences)

    chunks = []

    current_chunk = [sentences[0]]

    current_len = len(sentences[0].split())

    for i in range(1, len(sentences)):

        similarity = cosine_similarity(
            [embeddings[i]],
            [embeddings[i - 1]]
        )[0][0]

        sentence_len = len(sentences[i].split())

        # KEEP ADDING RELATED SENTENCES
        if (
            similarity >= similarity_threshold
            or len(current_chunk) < min_sentences
        ) and current_len + sentence_len <= max_words:

            current_chunk.append(sentences[i])

            current_len += sentence_len

        else:

            chunks.append(" ".join(current_chunk))

            current_chunk = [sentences[i]]

            current_len = sentence_len

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

Building vector db

In [ ]:
def build_vector_store(chunks):
    embeddings = embed_model.encode(chunks)
    embeddings = np.array(embeddings).astype('float32')
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index

Retrieval

In [ ]:
def retrieve(query,
             index,
             chunks,
             top_k=3):
    query_embedding = embed_model.encode([query])
    query_embedding = np.array(query_embedding).astype('float32')
    distances, indices = index.search(query_embedding, top_k)
    retrieved_chunks = [chunks[i] for i in indices[0]]
    return retrieved_chunks


Evaluation

In [ ]:
def evaluate_chunking(method_name,
                      chunk_function):
    print("\n")
    print("=" * 70)
    print(f"{method_name.upper()} CHUNKING")
    print("=" * 70)
    all_chunks = []

    # CREATE CHUNKS
    for doc in documents:
        chunks = chunk_function(doc)
        all_chunks.extend(chunks)
    print(f"\nTotal Chunks Created: {len(all_chunks)}")

    # AVERAGE CHUNK SIZE
    avg_len = np.mean([
        len(chunk.split())
        for chunk in all_chunks
    ])
    print(f"Average Chunk Length: {avg_len:.2f} words")

    # BUILD VECTOR STORE
    index = build_vector_store(all_chunks)
    top1_correct = 0
    top3_correct = 0
    
    # QUERY LOOP
    for query, expected_answer in queries:
        retrieved_chunks = retrieve(
            query,
            index,
            all_chunks,
            top_k=3
        )
        top1_hit = expected_answer.lower() in \
                   retrieved_chunks[0].lower()
        top3_hit = any(
            expected_answer.lower() in chunk.lower()
            for chunk in retrieved_chunks
        )
        if top1_hit:
            top1_correct += 1
        if top3_hit:
            top3_correct += 1

        print("\n--------------------------------------------------")
        print(f"Query: {query}")

        print(f"Expected Answer: {expected_answer}")

        print(f"Top-1 Correct: {top1_hit}")

        print(f"Top-3 Correct: {top3_hit}")

        print("\nTop Retrieved Chunk:")
        print(retrieved_chunks[0][:300])

    top1_accuracy = top1_correct / len(queries)
    top3_accuracy = top3_correct / len(queries)

    print("\n")
    print("=" * 70)

    print(f"TOP-1 ACCURACY : {top1_accuracy:.2f}")

    print(f"TOP-3 ACCURACY : {top3_accuracy:.2f}")

    print("=" * 70)

    return top1_accuracy, top3_accuracy

Results

In [ ]:
fixed_top1, fixed_top3 = evaluate_chunking(
    "Fixed Size",
    fixed_size_chunking
)

sentence_top1, sentence_top3 = evaluate_chunking(
    "Sentence Aware",
    sentence_aware_chunking
)

semantic_top1, semantic_top3 = evaluate_chunking(
    "Semantic",
    semantic_chunking
)



FIXED SIZE CHUNKING

Total Chunks Created: 21
Average Chunk Length: 21.05 words

--------------------------------------------------
Query: Which technology helps build neural network models?
Expected Answer: TensorFlow
Top-1 Correct: False
Top-3 Correct: True

Top Retrieved Chunk:
learning frameworks. They help researchers build neural network models efficiently. Climate change affects ecosystems and global temperatures. Carbon emissions are one of the major causes

--------------------------------------------------
Query: What reduces dependence on fossil fuels?
Expected Answer: Renewable energy
Top-1 Correct: True
Top-3 Correct: True

Top Retrieved Chunk:
one of the major causes of global warming. Renewable energy sources such as solar and wind power reduce dependence on fossil fuels. Governments are investing

--------------------------------------------------
Query: Who built the Taj Mahal?
Expected Answer: Shah Jahan
Top-1 Correct: True
Top-3 Correct: True

Top Retrieved Chunk:


Comparision

In [ ]:
print("\n")
print("=" * 70)
print("COMPARISON")
print("=" * 70)

print(f"Fixed-size     -> Top1: {fixed_top1:.2f} | Top3: {fixed_top3:.2f}")
print(f"Sentence-aware -> Top1: {sentence_top1:.2f} | Top3: {sentence_top3:.2f}")
print(f"Semantic       -> Top1: {semantic_top1:.2f} | Top3: {semantic_top3:.2f}")

print("=" * 70)



FINAL COMPARISON
Fixed-size     -> Top1: 0.70 | Top3: 1.00
Sentence-aware -> Top1: 0.90 | Top3: 1.00
Semantic       -> Top1: 1.00 | Top3: 1.00


### Query Rewriting

Methods:
1. Query Expansion
2. Query Relaxation
3. Query Segmentation
4. Query Scoping

In [25]:
from collections import defaultdict
import re

In [ ]:
documents = {

    1: {
        "title": "Intellectual Property Attorney",
        "body": "An attorney specializing in intellectual property law."
    },

    2: {
        "title": "IP Networking Basics",
        "body": "Internet protocol and computer networking concepts."
    },

    3: {
        "title": "Cute Fluffy Kittens",
        "body": "Photos of fluffy kittens and cute cats."
    },

    4: {
        "title": "Fluffy Kittens Playing",
        "body": "Small fluffy kittens playing together."
    },

    5: {
        "title": "White Dress Shirt",
        "body": "Formal white dress shirt for office wear."
    },

    6: {
        "title": "White Shirt Dress",
        "body": "Elegant white shirt dress for women."
    },

    7: {
        "title": "Machine Learning",
        "body": "Introduction to machine learning algorithms."
    },

    8: {
        "title": "Cute Puppies",
        "body": "Cute dogs and puppies."
    },

    9: {
        "title": "White Casual Shirt",
        "body": "Simple white shirt for casual wear."
    },

    10: {
        "title": "Dress Collection",
        "body": "Women dress collection and fashion."
    },

    11: {
        "title": "Learning Techniques",
        "body": "Different educational learning methods."
    },

    12: {
        "title": "Machine Repair",
        "body": "Repairing industrial machines."
    },

    13: {
        "title": "Learning Machine",
        "body": "A book about automated systems."
    },

    14: {
        "title": "Artificial Intelligence",
        "body": "Machine learning is widely used in AI."
    }
}

In [27]:
# ============================================================
# TOKENIZER
# ============================================================

def tokenize(text):
    return re.findall(r'\w+', text.lower())

In [ ]:
# ============================================================
# SEARCH FUNCTION
# ============================================================

def search(query_terms,
           docs,
           phrase=None,
           strict_and=False):

    scores = defaultdict(int)

    for doc_id, content in docs.items():

        title = content["title"].lower()
        body = content["body"].lower()

        searchable = title + " " + body

        matched_terms = 0

        # ----------------------------------------------------
        # NORMAL TERM MATCHING
        # ----------------------------------------------------

        for term in query_terms:

            if term.lower() in searchable:
                matched_terms += 1
                scores[doc_id] += 1

        # ----------------------------------------------------
        # STRICT AND MATCHING
        # ----------------------------------------------------

        if strict_and and matched_terms < len(query_terms):
            scores.pop(doc_id, None)
            continue

        # ----------------------------------------------------
        # QUERY SEGMENTATION LITE
        # ----------------------------------------------------

        if phrase:

            # exact phrase boost
            if phrase.lower() in searchable:
                scores[doc_id] += 50

            # penalize reversed phrase
            reversed_phrase = " ".join(
                phrase.lower().split()[::-1]
            )

            if reversed_phrase in searchable:
                scores[doc_id] -= 20

    # --------------------------------------------------------
    # SORT RESULTS
    # --------------------------------------------------------

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [doc for doc, score in ranked]

In [30]:
# ============================================================
# QUERY EXPANSION
# ============================================================

print("=" * 60)
print("QUERY EXPANSION")
print("=" * 60)

# baseline
baseline_results = search(
    ["ip", "lawer"],
    documents,
    strict_and=True
)

# expanded query
expanded_results = search(
    [
        "ip",
        "intellectual property",
        "lawer",
        "attorney"
    ],
    documents
)

print("\nOriginal Query: ip lawer")

print("\nBaseline Retrieved:")
print(baseline_results[:3])

print("\nExpanded Query:")
print("(ip OR intellectual property)")
print("(lawer OR attorney)")

print("\nRetrieved Documents:")
print(expanded_results[:3])

QUERY EXPANSION

Original Query: ip lawer

Baseline Retrieved:
[]

Expanded Query:
(ip OR intellectual property)
(lawer OR attorney)

Retrieved Documents:
[1, 2]


In [31]:
# ============================================================
# QUERY RELAXATION
# ============================================================

print("\n" + "=" * 60)
print("QUERY RELAXATION")
print("=" * 60)

# baseline
baseline_results = search(
    ["cute", "fluffy", "kittens"],
    documents,
    strict_and=True
)

# relaxed query
relaxed_results = search(
    ["fluffy", "kittens"],
    documents
)

print("\nOriginal Query: cute fluffy kittens")

print("\nBaseline Retrieved:")
print(baseline_results[:3])

print("\nRelaxed Query:")
print("fluffy kittens")

print("\nRetrieved Documents:")
print(relaxed_results[:3])


QUERY RELAXATION

Original Query: cute fluffy kittens

Baseline Retrieved:
[3]

Relaxed Query:
fluffy kittens

Retrieved Documents:
[3, 4]


In [32]:
# ============================================================
# QUERY SEGMENTATION
# ============================================================

print("\n" + "=" * 60)
print("QUERY SEGMENTATION")
print("=" * 60)

# baseline
baseline_results = search(
    ["white", "dress", "shirt"],
    documents
)

# segmented query
segmented_results = search(
    ["white"],
    documents,
    phrase="dress shirt"
)

print('\nOriginal Query: white dress shirt')

print("\nBaseline Retrieved:")
print(baseline_results[:3])

print('\nSegmented Query:')
print('white "dress shirt"')

print("\nRetrieved Documents:")
print(segmented_results[:3])


QUERY SEGMENTATION

Original Query: white dress shirt

Baseline Retrieved:
[5, 6, 9]

Segmented Query:
white "dress shirt"

Retrieved Documents:
[5, 9, 6]


In [35]:
# ============================================================
# QUERY SCOPING
# ============================================================

print("\n" + "=" * 60)
print("QUERY SCOPING")
print("=" * 60)

query_terms = ["machine", "learning"]

# ------------------------------------------------------------
# BASELINE SEARCH
# Searches title + body
# ------------------------------------------------------------

baseline_scores = defaultdict(int)

for doc_id, content in documents.items():

    title = content["title"].lower()
    body = content["body"].lower()

    for term in query_terms:

        # title match gets higher weight
        if term in title:
            baseline_scores[doc_id] += 3

        # body match gets lower weight
        if term in body:
            baseline_scores[doc_id] += 1

baseline_ranked = sorted(
    baseline_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

baseline_results = [
    doc for doc, score in baseline_ranked
]

# ------------------------------------------------------------
# SCOPED SEARCH
# Title-only matching
# ------------------------------------------------------------

scoped_scores = defaultdict(int)

for doc_id, content in documents.items():

    title = content["title"].lower()

    for term in query_terms:

        if term in title:
            scoped_scores[doc_id] += 1

scoped_ranked = sorted(
    scoped_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

scoped_results = [
    doc for doc, score in scoped_ranked
]

print("\nOriginal Query: machine learning")

print("\nBaseline Retrieved:")
print(baseline_results[:5])

print("\nScoped Query:")
print("Search only in titles/headings")

print("\nRetrieved Documents:")
print(scoped_results[:5])


QUERY SCOPING

Original Query: machine learning

Baseline Retrieved:
[7, 13, 11, 12, 14]

Scoped Query:
Search only in titles/headings

Retrieved Documents:
[7, 13, 11, 12]
